# 07 · Anahtar Kelime Şablonu ve Sözlükle OCR Düzeltme

**15 Ekim danışman görüşmesi:**

| İstek | Danışmanın ifadesi (özet) | Bölüm |
|---|---|---|
| 4 · Şablon | Her türün hemen her belgesinde bulunan ortak ifadeler (faturada fiyat, şirket adı, miktar; dilekçede "sayın, arz ederim") OCR metninde aranıp sınıflandırmaya **destek** olsun | A |
| 5 · Sözlük | OCR'ın yanlış okuduğu kelimeler ("Dater" → "Date") sözlükten düzeltilsin | B |

Veri seti İngilizce olduğu için şablon ifadeleri İngilizcedir ("sayın / arz ederim" → "Dear / Sincerely").
Sınıf listesi `siniflar.json`'dan okunur (06'nın son hücresi). Süre: ~5 dk, GPU gerekmez.

> Girdi: `data/cleaned_dataset.csv` (03) ya da OCR çıktısı (02 + 06). Çıktılar `ROOT/` ve `figures/` altına `sablon_*`, `sozluk_*` adıyla kaydedilir.

## 1. Kurulum

In [ ]:
!pip -q install symspellpy

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys, json, importlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
SEED = 42
CFG_FILE = ROOT / "siniflar.json"
CLASSES = json.loads(CFG_FILE.read_text())["CLASSES"] if CFG_FILE.exists() else ["email", "form", "invoice", "resume"]
print("Sınıflar:", CLASSES, "(siniflar.json)" if CFG_FILE.exists() else "(varsayılan)")

Modüller Drive'a yazılır (05 demo da aynı modülleri kullanır).

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/sablon.py
"""Belge türü şablonları (danışman isteği, 15 Ekim): her türün hemen her belgesinde bulunan ortak ifadeler.

OCR metninde bir türün şablon ifadeleri bulunuyorsa bu, makine öğrenmesi modelinin kararını destekleyen
ikinci bir kanıttır (ör. faturada "invoice", "total", para tutarı; mektupta "Dear", "Sincerely").
Veri seti İngilizce olduğu için ifadeler İngilizcedir; Türkçe karşılıkları yorumda verilmiştir.
"""
import re

# her ifade: (kısa ad, düzenli ifade) · metin küçük harfe çevrilmiş temiz OCR metnine uygulanır
TEMPLATES = {
    "email": [  # e-posta başlık alanları
        ("from:", r"^\s*from\s*[:;]"),
        ("to:", r"^\s*to\s*[:;]"),
        ("subject:", r"^\s*(?:subject|subj)\s*[:;]"),
        ("sent/date:", r"^\s*(?:sent|date)\s*[:;]"),
        ("cc:", r"^\s*b?cc\s*[:;]"),
        ("@ adresi", r"[\w.+-]+@[\w-]+"),
        ("message", r"\bmessage\b"),
    ],
    "invoice": [  # fatura: numara, tutar, miktar, ödeme
        ("invoice", r"\binvoice\b"),
        ("total", r"\btotal\b"),
        ("amount", r"\bamount\b"),
        ("para tutarı", r"[$€£]\s?\d|\b\d{1,3}(?:,\d{3})*\.\d{2}\b"),
        ("qty/quantity", r"\b(?:qty|quantity)\b"),
        ("price", r"\bprice\b"),
        ("bill/ship/sold to", r"\b(?:bill|ship|sold|remit)\s*to\b"),
        ("payment/due", r"\b(?:payment|payable|due|terms)\b"),
    ],
    "resume": [  # CV bölüm başlıkları
        ("education", r"\beducation\b"),
        ("experience", r"\b(?:experience|employment)\b"),
        ("university/college", r"\b(?:university|college|institute)\b"),
        ("degree", r"\b(?:b\.?s\.?|m\.?s\.?|ph\.?d\.?|bachelor|master|doctor|degree)\b"),
        ("born/birth", r"\b(?:born|birth|married)\b"),
        ("research", r"\bresearch\b"),
        ("publications/references", r"\b(?:publications|references|honors|awards)\b"),
    ],
    "form": [  # form: faks kapağı, doldurulacak alanlar
        ("fax/facsimile", r"\b(?:fax|facsimile|telecopy|telecopier)\b"),
        ("transmission", r"\btransmi(?:ssion|tted|t)\b"),
        ("pages", r"\bpages?\b"),
        ("name:", r"\bname\s*[:;]"),
        ("date:", r"\bdate\s*[:;]"),
        ("signature/approved", r"\b(?:signature|signed|approv(?:ed|al))\b"),
        ("request/number", r"\b(?:request|number|no\.)\b"),
    ],
    "letter": [  # mektup / resmi yazı (dilekçe): hitap, kapanış, ek
        ("dear", r"^\s*dear\b"),
        ("sincerely/regards", r"\b(?:sincerely|yours\s+truly|very\s+truly|regards|respectfully|cordially)\b"),
        ("re:", r"^\s*re\s*[:;]"),
        ("enclosed/enclosure", r"\benclos(?:ed|ure)\b"),
        ("thank you", r"\bthank\s+you\b"),
        ("letter", r"\bletter\b"),
        ("please", r"\bplease\b"),
    ],
    "news_article": [  # gazete haberi
        ("said", r"\bsaid\b"),
        ("ajans (AP/UPI/Reuters)", r"\((?:ap|upi)\)|\breuters\b|\bassociated press\b"),
        ("percent", r"\bpercent\b"),
        ("according to", r"\baccording to\b"),
        ("spokesman/officials", r"\b(?:spokesman|spokeswoman|officials?)\b"),
        ("yesterday/today", r"\b(?:yesterday|today|last week)\b"),
        ("times/post/journal", r"\b(?:times|post|journal|tribune|herald|news)\b"),
    ],
    "scientific_publication": [  # bilimsel makale bölümleri
        ("abstract", r"\babstract\b"),
        ("introduction", r"\bintroduction\b"),
        ("methods/results", r"\b(?:methods?|results)\b"),
        ("references", r"\breferences\b"),
        ("et al.", r"\bet\s+al\b"),
        ("fig./table", r"\b(?:fig|figure|table)\b"),
        ("journal/vol.", r"\b(?:journal|vol|volume)\b"),
    ],
}

_COMPILED = {c: [(n, re.compile(p, re.I | re.M)) for n, p in pats] for c, pats in TEMPLATES.items()}


def match_template(text, classes=None):
    """{sınıf: [eşleşen ifade adları]}"""
    text = text or ""
    return {c: [n for n, rx in _COMPILED[c] if rx.search(text)] for c in (classes or TEMPLATES)}


def template_scores(text, classes=None):
    """{sınıf: eşleşen ifade oranı (0-1)}"""
    m = match_template(text, classes)
    return {c: len(v) / len(TEMPLATES[c]) for c, v in m.items()}


def template_vector(text, classes):
    """Model için özellik vektörü: her ifade için 0/1 + her sınıf için eşleşme oranı."""
    m = match_template(text, classes)
    bits = [float(n in m[c]) for c in classes for n, _ in _COMPILED[c]]
    return bits + [len(m[c]) / len(TEMPLATES[c]) for c in classes]


def template_feature_names(classes):
    return [f"{c}:{n}" for c in classes for n, _ in _COMPILED[c]] + [f"{c}:oran" for c in classes]


def rule_predict(text, classes):
    """Sadece şablona göre tahmin: en yüksek eşleşme oranına sahip sınıf (eşleşme yoksa None)."""
    s = template_scores(text, classes)
    best = max(s, key=s.get)
    return best if s[best] > 0 else None


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_fix.py
"""Sözlük tabanlı OCR hata düzeltme (danışman isteği, 15 Ekim: "Dater" → "Date" gibi hatalar).

Üç adım, sırayla:
1. OCR karışıklık kuralları: Tesseract'ın sık karıştırdığı harf grupları (rn↔m, 1↔l, 0↔o, vv↔w, cl↔d ...).
   Kelime sözlükte yoksa ve bir kural uygulanınca sözlükte bir kelimeye dönüşüyorsa o kelime alınır.
2. Alan adı kuralı: satır başındaki kelime bilinen bir alan adına (Date, From, To, Total ...) 1 harf
   uzaklıktaysa düzeltilir. "Dater" sözlükte geçerli bir kelime olduğu için sadece bu kural yakalayabilir.
3. SymSpell: sözlükte olmayan kelime, sıklık sözlüğündeki en yakın kelimeye çevrilir
   (5 harfe kadar en fazla 1, daha uzunlarda en fazla 2 harf farkı).

Rakam, sembol içeren kelimeler ve 3 harften kısa kelimeler değiştirilmez.
Kişi/firma adları da sözlükte olmadığı için yanlış "düzeltilebilir"; bu yüzden düzeltilmiş metin
sadece sınıflandırma için kullanılır, içerik çıkarımı (ad, firma) orijinal metinden yapılır.
"""
import re
from collections import Counter
from importlib import resources

from symspellpy import SymSpell, Verbosity

_SYM = None
_WORDS = None
OCR_CONFUSIONS = [("rn", "m"), ("m", "rn"), ("1", "l"), ("l", "1"), ("0", "o"), ("vv", "w"), ("cl", "d"),
                  ("ii", "u"), ("li", "h"), ("5", "s"), ("8", "b"), ("|", "l"), ("l", "i"), ("i", "l")]
FIELD_NAMES = ["date", "from", "to", "subject", "name", "address", "total", "amount", "invoice", "phone",
               "fax", "pages", "re", "cc", "title", "company", "signature"]
_TOKEN = re.compile(r"[A-Za-z0-9|]+")


def _load():
    global _SYM, _WORDS
    if _SYM is None:
        _SYM = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
        path = resources.files("symspellpy") / "frequency_dictionary_en_82_765.txt"
        _SYM.load_dictionary(str(path), term_index=0, count_index=1)
        _WORDS = _SYM.words
    return _SYM


def in_dictionary(word):
    _load()
    return word.lower() in _WORDS


def _match_case(src, dst):
    if src.isupper():
        return dst.upper()
    if src[:1].isupper():
        return dst[:1].upper() + dst[1:]
    return dst


def _confusion_fix(low):
    for a, b in OCR_CONFUSIONS:
        if a in low:
            cand = low.replace(a, b)
            if cand in _WORDS:
                return cand
    return None


def correct_word(word, line_start=False):
    """(düzeltilmiş kelime, kural adı) — değişiklik yoksa (word, None)."""
    sym = _load()
    low = word.lower()
    if line_start:
        for f in FIELD_NAMES:
            if len(f) >= 3 and low != f and len(low) == len(f) + 1 and low.startswith(f):
                return _match_case(word, f), "alan adı"
    if len(low) < 3 or low in _WORDS:
        return word, None
    fixed = _confusion_fix(low)
    if fixed:
        return _match_case(word, fixed), "OCR karışıklığı"
    if not low.isalpha():
        return word, None
    max_d = 1 if len(low) <= 5 else 2
    sug = sym.lookup(low, Verbosity.TOP, max_edit_distance=max_d)
    if sug and sug[0].term != low:
        return _match_case(word, sug[0].term), "SymSpell"
    return word, None


def correct_text(text, log=None):
    """Metindeki kelimeleri düzeltir. `log` bir Counter ise (orijinal, düzeltme, kural) sayılır."""
    if not text:
        return text
    out_lines = []
    for line in text.split("\n"):
        pieces, last, first = [], 0, True
        for m in _TOKEN.finditer(line):
            w = m.group(0)
            new, rule = correct_word(w, line_start=first and m.start() <= 2)
            first = False
            pieces.append(line[last:m.start()])
            pieces.append(new)
            last = m.end()
            if rule and log is not None:
                log[(w, new, rule)] += 1
        pieces.append(line[last:])
        out_lines.append("".join(pieces))
    return "\n".join(out_lines)


def dict_ratio(text):
    """Harflerden oluşan (≥3) kelimelerin sözlükte bulunma oranı."""
    _load()
    toks = [t.lower() for t in re.findall(r"[A-Za-z]+", text or "") if len(t) >= 3]
    return sum(t in _WORDS for t in toks) / len(toks) if toks else 0.0


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/text_clean.py
import re
import unicodedata

# OCR'ın sık ürettiği tipografik karakterleri sadeleştir
_CHAR_MAP = str.maketrans({
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "•": "-", "·": "-",
    " ": " ", "­": "",
})
_WORDISH = re.compile(r"[A-Za-z0-9]")
_NOISE_TOKEN = re.compile(r"^[^\w$%@&#+]+$")          # sadece sembol: | ~ ` ' . , _ vb.
_EDGE_JUNK_L = re.compile(r"^[|_~`'\"\s]+")
_EDGE_JUNK_R = re.compile(r"[|_~`\s]+$")
_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
_URL = re.compile(r"\b(?:https?://|www\.)\S+", re.I)
_MONEY = re.compile(r"[$€£]\s?\d[\d,]*(?:\.\d+)?")
_NUM = re.compile(r"\d+")


def alnum_ratio(s):
    s = s.replace(" ", "")
    return sum(c.isalnum() for c in s) / len(s) if s else 0.0


def clean_text(raw, min_line_alnum=0.4):
    """OCR metnini temizler; büyük/küçük harf, e-posta, tutar gibi bilgiler korunur
    (bilgi çıkarımı aşamasında gerekecek)."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    t = unicodedata.normalize("NFKC", raw).translate(_CHAR_MAP)
    t = "".join(c for c in t if c == "\n" or c.isprintable())
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)               # satır sonu tirelemesini birleştir
    lines = []
    for line in t.split("\n"):
        tokens = [tok for tok in line.split() if not _NOISE_TOKEN.match(tok)]
        line = " ".join(tokens)
        line = _EDGE_JUNK_R.sub("", _EDGE_JUNK_L.sub("", line))
        if len(line) < 2 or not _WORDISH.search(line):
            continue
        if alnum_ratio(line) < min_line_alnum:              # çoğu sembol olan çöp satır
            continue
        lines.append(line)
    return "\n".join(lines)


def normalize_for_model(clean):
    """Sınıflandırıcı için: küçük harf + e-posta/URL/tutar/sayıları ortak simgeye çevir.
    Böylece model 'john@acme.com' yerine 'bir e-posta adresi var' bilgisini öğrenir."""
    t = clean.lower()
    t = _EMAIL.sub(" emailaddr ", t)
    t = _URL.sub(" urladdr ", t)
    t = _MONEY.sub(" moneyamt ", t)
    t = _NUM.sub("0", t)
    t = re.sub(r"[^a-z0\s]", " ", t)
    return " ".join(t.split())


def garbage_ratio(raw):
    """Ham OCR çıktısındaki anlamsız token oranı (harf/rakam içermeyen ya da
    harf-rakam-sembol karışık kısa parçalar)."""
    toks = raw.split() if isinstance(raw, str) else []
    if not toks:
        return 1.0
    bad = 0
    for tok in toks:
        if not _WORDISH.search(tok):
            bad += 1
        elif len(tok) <= 3 and alnum_ratio(tok) < 0.67:
            bad += 1
    return bad / len(toks)


In [ ]:
sys.path.insert(0, str(ROOT))
import text_clean, sablon, ocr_fix
for m in (text_clean, sablon, ocr_fix): importlib.reload(m)
from text_clean import clean_text, normalize_for_model


def load_ocr_data():
    """02'nin OCR çıktısı + 06'da OCR'lanan aday sınıflar → sadece CLASSES'taki sınıflar."""
    parts = [pd.read_csv(DATA / "ocr_dataset.csv")]
    if (DATA / "ocr_dataset_aday.csv").exists():
        parts.append(pd.read_csv(DATA / "ocr_dataset_aday.csv"))
    d = pd.concat(parts, ignore_index=True).drop_duplicates("path")
    d = d[d["label"].isin(CLASSES)].reset_index(drop=True)
    d["text"] = d["text"].fillna("")
    missing = set(CLASSES) - set(d["label"])
    assert not missing, f"OCR verisi olmayan sınıf: {missing} → önce 06_sinif_secimi'ni çalıştırın."
    return d


df = pd.read_csv(DATA / "cleaned_dataset.csv") if (DATA / "cleaned_dataset.csv").exists() else None
if df is not None and set(CLASSES) <= set(df["label"]):
    df = df[df["label"].isin(CLASSES)].reset_index(drop=True)
else:
    print("cleaned_dataset.csv yok ya da sınıflar değişmiş → OCR çıktısından temizleniyor")
    df = load_ocr_data()
    df["text_clean"] = df["text"].map(clean_text)
    df["text_norm"] = df["text_clean"].map(normalize_for_model)
    df["is_empty"] = df["text_clean"].str.split().str.len().fillna(0) < 5
df["text_clean"] = df["text_clean"].fillna("")
df["text_norm"] = df["text_norm"].fillna("")
train = df[(df.split == "train") & ~df.is_empty].copy()
val, test = df[df.split == "validation"].copy(), df[df.split == "test"].copy()
print(f"Train {len(train)} · Validation {len(val)} · Test {len(test)}")
pd.crosstab(df.label, df.split)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay
from scipy.stats import binomtest


def char_tfidf():      # 04'teki en iyi özellik
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.9, sublinear_tf=True, max_features=100_000)


def word_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True,
                           stop_words="english", token_pattern=r"(?u)\b[a-z0][a-z0]+\b")


def scores(y, p):
    _, _, f, _ = precision_recall_fscore_support(y, p, labels=CLASSES, zero_division=0)
    return {"accuracy": accuracy_score(y, p), "macro_f1": f1_score(y, p, average="macro"),
            **{f"F1_{c}": v for c, v in zip(CLASSES, f)}}


def mcnemar(y, pa, pb):
    ca, cb = (pa == y), (pb == y)
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return n01, n10, (binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0)


def fit_svm(Xtr, ytr, Xva, yva, grid=(0.05, 0.1, 0.5, 1, 5)):
    """C değeri validation macro-F1'e göre seçilir."""
    best = None
    for C in grid:
        clf = LinearSVC(C=C, random_state=SEED).fit(Xtr, ytr)
        f = f1_score(yva, clf.predict(Xva), average="macro")
        if best is None or f > best[1]:
            best = (clf, f, C)
    return best

# A · Anahtar kelime şablonu (istek 4)

## A1. Şablon: her türün ortak ifadeleri

Şablon elle, belge türlerinin bilinen yapısından hazırlandı (fatura → `invoice`, `total`, tutar; mektup → `Dear`, `Sincerely` ...).
A2'de her ifadenin gerçekten o türün belgelerinin ne kadarında geçtiği **veriyle** kontrol edilir.

In [ ]:
from sablon import TEMPLATES, match_template, template_scores, template_vector, template_feature_names, rule_predict

tpl = pd.DataFrame([{"sınıf": c, "ifade": n, "düzenli ifade": p} for c in CLASSES for n, p in TEMPLATES[c]])
tpl

## A2. Şablon ifadeleri belgelerin ne kadarında geçiyor?

Satır: şablon ifadesi · Sütun: gerçek belge türü · Değer: o türdeki **train** belgelerinin yüzde kaçında ifade geçiyor.
İyi bir şablon ifadesi kendi sınıfında **yüksek**, diğer sınıflarda **düşük** olmalı.

In [ ]:
low_train = train["text_clean"].str.lower()
match_lists = low_train.map(lambda t: match_template(t, CLASSES))
rows = []
for c in CLASSES:
    for n, _ in TEMPLATES[c]:
        has = match_lists.map(lambda m: n in m[c])
        rows.append({"şablon": c, "ifade": n, **{k: 100 * has[train.label == k].mean() for k in CLASSES}})
cov = pd.DataFrame(rows).set_index(["şablon", "ifade"]).round(1)
cov.to_csv(ROOT / "sablon_kapsama.csv")

fig, ax = plt.subplots(figsize=(1.6 * len(CLASSES) + 4, 0.38 * len(cov) + 1.5))
im = ax.imshow(cov.values, cmap="Greens", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(len(cov))); ax.set_yticklabels([f"{s} · {i}" for s, i in cov.index], fontsize=9)
for i in range(cov.shape[0]):
    for j in range(cov.shape[1]):
        v = cov.values[i, j]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 60 else "black")
ax.set_xlabel("Gerçek belge türü (train)"); ax.set_title("Şablon ifadelerinin geçtiği belge oranı (%)")
plt.colorbar(im, ax=ax, fraction=0.03); plt.tight_layout(); plt.savefig(FIG / "sablon_kapsama.png", dpi=120); plt.show()

# Her türde: en az 1 / en az 2 şablon ifadesi geçen belge oranı
own = pd.DataFrame({
    "en az 1 ifade (%)": [100 * match_lists[train.label == c].map(lambda m: len(m[c]) >= 1).mean() for c in CLASSES],
    "en az 2 ifade (%)": [100 * match_lists[train.label == c].map(lambda m: len(m[c]) >= 2).mean() for c in CLASSES],
    "ort. eşleşen ifade": [match_lists[train.label == c].map(lambda m: len(m[c])).mean() for c in CLASSES],
}, index=CLASSES).round(1)
own

## A3. Veriden çıkan ortak kelimeler

Şablonu kontrol etmek için aynı soru veriye soruldu: **hangi kelimeler bir türün belgelerinin çoğunda geçiyor ama diğer türlerde az geçiyor?**
Skor = (sınıf içinde geçtiği belge oranı) − (diğer sınıflardaki en yüksek oran).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(binary=True, token_pattern=r"(?u)\b[a-z]{3,}\b", min_df=10)
Xb = cv.fit_transform(train["text_norm"])
vocab = np.array(cv.get_feature_names_out())
dfreq = pd.DataFrame({c: np.asarray(Xb[(train.label == c).values].mean(0)).ravel() for c in CLASSES}, index=vocab)
rows = []
for c in CLASSES:
    others = dfreq.drop(columns=c).max(axis=1)
    s = (dfreq[c] - others).sort_values(ascending=False).head(10)
    for w, v in s.items():
        rows.append({"sınıf": c, "kelime": w, "sınıfta (%)": round(100 * dfreq.loc[w, c], 1),
                     "diğerlerinde en çok (%)": round(100 * others[w], 1), "şablonda": any(w in n for n, _ in TEMPLATES[c])})
data_kw = pd.DataFrame(rows)
data_kw.to_csv(ROOT / "sablon_veriden_kelimeler.csv", index=False)
display(data_kw.pivot_table(index=data_kw.groupby("sınıf").cumcount() + 1, columns="sınıf", values="kelime", aggfunc="first")[CLASSES])
data_kw

## A4. Sadece şablonla sınıflandırma (kural tabanlı)

Makine öğrenmesi olmadan: test belgesinde en yüksek oranda şablon ifadesi eşleşen tür seçilir. Hiç eşleşme yoksa tahmin "yok" sayılır (yanlış kabul edilir).
Bu, şablonun tek başına ne kadar bilgi taşıdığını gösterir.

In [ ]:
test_low = test["text_clean"].str.lower()
rule_pred = test_low.map(lambda t: rule_predict(t, CLASSES)).fillna("eşleşme yok").values
rule_res = scores(test.label.values, rule_pred)
print(f"Sadece şablon · test accuracy {rule_res['accuracy']:.4f} · macro-F1 {rule_res['macro_f1']:.4f}")
print(f"Hiç şablon ifadesi bulunmayan test belgesi: {(rule_pred == 'eşleşme yok').sum()} / {len(test)}")
# Satır: gerçek tür · sütun: şablonun tahmini (+ hiç ifade bulunamayanlar için "eşleşme yok" sütunu)
labs = CLASSES + (["eşleşme yok"] if (rule_pred == "eşleşme yok").any() else [])
cm = confusion_matrix(test.label, rule_pred, labels=labs)[:len(CLASSES)]
fig, ax = plt.subplots(figsize=(1.5 * len(labs) + 1.5, 1.2 * len(CLASSES) + 1.5))
ax.imshow(cm, cmap="Oranges")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=30, ha="right")
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Şablonun tahmini"); ax.set_ylabel("Gerçek tür")
ax.set_title(f"Sadece şablon · macro-F1 = {rule_res['macro_f1']:.3f}")
plt.tight_layout(); plt.savefig(FIG / "sablon_kural_confusion.png", dpi=120); plt.show()

## A5. Şablon, modele destek olarak (TF-IDF + şablon)

Danışmanın istediği kullanım: şablon **ikinci bir kanıt**. Her belge için şablon ifadelerinin var/yok bilgisi ve sınıf başına eşleşme oranı,
karakter TF-IDF vektörüne eklenir (04'teki tablo özelliği deneyiyle aynı yöntem). Şablonun ağırlığı validation ile seçilir; test bir kez kullanılır.

In [ ]:
from scipy.sparse import hstack, csr_matrix

def tvec(frame):
    return np.array([template_vector(t, CLASSES) for t in frame["text_clean"].str.lower()])

T_tr, T_va, T_te = tvec(train), tvec(val), tvec(test)
cvec = char_tfidf().fit(train.text_norm)
X_tr, X_va, X_te = cvec.transform(train.text_norm), cvec.transform(val.text_norm), cvec.transform(test.text_norm)

base_clf, base_val, C0 = fit_svm(X_tr, train.label, X_va, val.label)
p_base = base_clf.predict(X_te)

def with_t(X, T, w):
    return hstack([X, csr_matrix(w * T)]).tocsr()

val_w = {}
for w in [0.05, 0.1, 0.2, 0.3, 0.5]:
    clf = LinearSVC(C=C0, random_state=SEED).fit(with_t(X_tr, T_tr, w), train.label)
    val_w[w] = f1_score(val.label, clf.predict(with_t(X_va, T_va, w)), average="macro")
best_w = max(val_w, key=val_w.get)
print("Validation macro-F1 · TF-IDF:", round(base_val, 4), "· şablon ağırlığı →", {k: round(v, 4) for k, v in val_w.items()}, "→ seçilen:", best_w)
hyb = LinearSVC(C=C0, random_state=SEED).fit(with_t(X_tr, T_tr, best_w), train.label)
p_hyb = hyb.predict(with_t(X_te, T_te, best_w))

tpl_res = pd.DataFrame([{"yöntem": "Sadece şablon (kural)", **rule_res},
                        {"yöntem": "TF-IDF + Linear SVM", **scores(test.label.values, p_base)},
                        {"yöntem": "TF-IDF + şablon + Linear SVM", **scores(test.label.values, p_hyb)}]).round(4)
tpl_res.to_csv(ROOT / "sablon_sonuclari.csv", index=False)
display(tpl_res)
n01, n10, pv = mcnemar(test.label.values, p_base, p_hyb)
print(f"McNemar: sadece TF-IDF doğru {n01} · sadece TF-IDF+şablon doğru {n10} · p = {pv:.4f}",
      "(anlamlı)" if pv < 0.05 else "(400 belgede anlamlı değil)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(CLASSES)); w = 0.27
for k, (name, col) in enumerate([("Sadece şablon (kural)", "#e0a458"), ("TF-IDF + Linear SVM", "#9fb8b4"), ("TF-IDF + şablon + Linear SVM", "#0f6e66")]):
    r = tpl_res.set_index("yöntem").loc[name]
    bars = ax.bar(x + (k - 1) * w, [r[f"F1_{c}"] for c in CLASSES], w, label=f"{name} (macro {r.macro_f1:.3f})", color=col)
    ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=1)
ax.set_xticks(x); ax.set_xticklabels(CLASSES); ax.set_ylim(0, 1.08); ax.set_ylabel("Test F1")
ax.legend(fontsize=8, loc="lower left"); ax.grid(axis="y", alpha=.3); ax.set_title("Şablonun sınıf bazında katkısı")
plt.tight_layout(); plt.savefig(FIG / "sablon_katki.png", dpi=130); plt.show()

In [ ]:
# Şablonun düzelttiği / bozduğu örnekler
fixed = test[(p_base != test.label.values) & (p_hyb == test.label.values)]
broken = test[(p_base == test.label.values) & (p_hyb != test.label.values)]
print(f"Şablon sayesinde doğru olan: {len(fixed)} · şablon yüzünden yanlış olan: {len(broken)}")
for _, r in fixed.head(5).iterrows():
    m = match_template(r.text_clean.lower(), CLASSES)
    i = test.index.get_loc(r.name)
    print(f"- {Path(r.path).name}: gerçek {r.label} · TF-IDF {p_base[i]} → TF-IDF+şablon {p_hyb[i]} · eşleşen: {m[r.label]}")

# B · Sözlükle OCR düzeltme (istek 5)

`ocr_fix.correct_text` üç adımda düzeltir:

| Adım | Ne yapar | Örnek |
|---|---|---|
| OCR karışıklık kuralları | Tesseract'ın sık karıştırdığı harfler (rn↔m, 1↔l, 0↔o, cl↔d ...) değiştirilince sözlükte bir kelime çıkıyorsa onu alır | `resurne` → `resume`, `Contrlbution` → `Contribution` |
| Alan adı kuralı | Satır başındaki kelime bilinen bir alan adından 1 harf fazlaysa düzeltir | `Dater` → `Date` (Dater sözlükte geçerli bir kelime olduğu için sadece bu kural yakalar) |
| SymSpell (sıklık sözlüğü, 82.765 kelime) | Sözlükte olmayan kelimeyi en yakın sık kelimeye çevirir (≤5 harf: en fazla 1, uzun: en fazla 2 harf farkı) | `Anount` → `Amount`, `invoce` → `invoice` |

**Sınır:** Kişi ve firma adları sözlükte olmadığı için yanlış "düzeltilebilir" (ör. `Sena` → `Send`). Bu yüzden düzeltilmiş metin sadece **sınıflandırma** için kullanılır;
içerik çıkarımı (ad, firma, tutar) orijinal OCR metninden yapılır.

In [ ]:
from collections import Counter
from tqdm.auto import tqdm
from ocr_fix import correct_text, dict_ratio

log = Counter()
t0 = time.time()
df["text_fix"] = [correct_text(t, log) for t in tqdm(df["text_clean"])]
df["text_norm_fix"] = df["text_fix"].map(normalize_for_model)
print(f"Süre: {time.time() - t0:.0f} sn · toplam düzeltme: {sum(log.values())}")
rule_counts = pd.Series({r: sum(v for (o, n, rr), v in log.items() if rr == r) for r in ["OCR karışıklığı", "alan adı", "SymSpell"]})
display(rule_counts.rename("düzeltme sayısı").to_frame())

ex = pd.DataFrame([{"orijinal": o, "düzeltilmiş": n, "kural": r, "adet": v} for (o, n, r), v in log.most_common(40)])
ex.to_csv(ROOT / "sozluk_duzeltme_ornekleri.csv", index=False)
ex.head(30)

In [ ]:
# Sözlükte bulunan kelime oranı: düzeltmeden önce / sonra (sınıf bazında)
df["dict_before"] = df["text_clean"].map(dict_ratio)
df["dict_after"] = df["text_fix"].map(dict_ratio)
dr = df.groupby("label")[["dict_before", "dict_after"]].mean().reindex(CLASSES)
dr.loc["ortalama"] = df[["dict_before", "dict_after"]].mean()
dr = (100 * dr).round(1).rename(columns={"dict_before": "önce (%)", "dict_after": "sonra (%)"})
dr["fark"] = dr["sonra (%)"] - dr["önce (%)"]
display(dr)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(dr)); w = 0.38
b1 = ax.bar(x - w / 2, dr["önce (%)"], w, label="düzeltmeden önce", color="#9fb8b4")
b2 = ax.bar(x + w / 2, dr["sonra (%)"], w, label="sözlükle düzeltme sonrası", color="#0f6e66")
for b in (b1, b2): ax.bar_label(b, fmt="%.1f", fontsize=8, padding=2)
ax.set_xticks(x); ax.set_xticklabels(dr.index); ax.set_ylabel("Sözlükte bulunan kelime (%)")
ax.set_ylim(0, 105); ax.legend(loc="lower right"); ax.grid(axis="y", alpha=.3)
ax.set_title("Sözlükle düzeltmenin OCR metnine etkisi")
plt.tight_layout(); plt.savefig(FIG / "sozluk_oncesi_sonrasi.png", dpi=130); plt.show()

In [ ]:
# Önce / sonra örnek satırlar (her sınıftan bir belge)
import difflib
for c in CLASSES:
    pool = df[(df.label == c) & (df.split == "validation") & (df.text_clean != df.text_fix)]
    if pool.empty:
        continue
    r = pool.sample(1, random_state=SEED).iloc[0]
    print("=" * 90); print(f"[{c}] {Path(r.path).name}")
    shown = 0
    for a, b in zip(r.text_clean.splitlines(), r.text_fix.splitlines()):
        if a != b and shown < 4:
            print("  önce :", a[:110]); print("  sonra:", b[:110]); shown += 1

## B2. Düzeltme sınıflandırmayı etkiliyor mu?

Aynı modeller (kelime ve karakter TF-IDF + Linear SVM) düzeltilmemiş ve düzeltilmiş metinle eğitilir.
Kelime TF-IDF'te etkisi daha büyük beklenir: `Anount` ile `Amount` farklı kelime sayılır, karakter parçalarında ise zaten ortak parçaları vardır.

In [ ]:
train_f = df.loc[train.index]; val_f = df.loc[val.index]; test_f = df.loc[test.index]
rows = []
for fname, fmake in [("kelime", word_tfidf), ("karakter", char_tfidf)]:
    for col, tag in [("text_norm", "düzeltmesiz"), ("text_norm_fix", "sözlükle düzeltilmiş")]:
        v = fmake().fit(train_f[col])
        clf, fval, C = fit_svm(v.transform(train_f[col]), train_f.label, v.transform(val_f[col]), val_f.label)
        p = clf.predict(v.transform(test_f[col]))
        rows.append({"özellik": fname, "metin": tag, "C": C, "val_macro_f1": fval, **scores(test_f.label.values, p), "_p": p})
fix_res = pd.DataFrame([{k: v for k, v in r.items() if k != "_p"} for r in rows]).round(4)
fix_res.to_csv(ROOT / "sozluk_sonuclari.csv", index=False)
display(fix_res)
for fname in ["kelime", "karakter"]:
    a, b = [r["_p"] for r in rows if r["özellik"] == fname]
    n01, n10, pv = mcnemar(test_f.label.values, a, b)
    print(f"{fname:8s} TF-IDF · McNemar: sadece düzeltmesiz doğru {n01} · sadece düzeltilmiş doğru {n10} · p = {pv:.4f}")

In [ ]:
# Düzeltilmiş metin kaydedilir (05 demo ve sonraki deneyler için)
try:
    df[["path", "label", "split", "text_fix", "text_norm_fix"]].to_csv(DATA / "ocr_duzeltilmis.csv", index=False)
    print("Kaydedildi:", DATA / "ocr_duzeltilmis.csv")
except OSError as e:
    print("Kaydedilemedi (Drive dolu olabilir):", e)

## Özet (rapor için)

In [ ]:
best_t = tpl_res.set_index("yöntem")
summary = {
    "Sınıflar": ", ".join(CLASSES),
    "Sadece şablon · macro-F1": round(best_t.loc["Sadece şablon (kural)", "macro_f1"], 4),
    "TF-IDF · macro-F1": round(best_t.loc["TF-IDF + Linear SVM", "macro_f1"], 4),
    "TF-IDF + şablon · macro-F1": round(best_t.loc["TF-IDF + şablon + Linear SVM", "macro_f1"], 4),
    "Sözlük: toplam düzeltme": int(sum(log.values())),
    "Sözlükte bulunan kelime (önce → sonra)": f"%{dr.loc['ortalama', 'önce (%)']} → %{dr.loc['ortalama', 'sonra (%)']}",
    "Kelime TF-IDF macro-F1 (düzeltmesiz → düzeltilmiş)": " → ".join(f"{v:.4f}" for v in fix_res[fix_res.özellik == "kelime"].macro_f1),
    "Karakter TF-IDF macro-F1 (düzeltmesiz → düzeltilmiş)": " → ".join(f"{v:.4f}" for v in fix_res[fix_res.özellik == "karakter"].macro_f1),
}
pd.Series(summary).to_frame("değer")

## Çıktılar

| Dosya | İçerik |
|---|---|
| `sablon_kapsama.csv`, `figures/sablon_kapsama.png` | Şablon ifadelerinin her türde geçtiği belge oranı |
| `sablon_veriden_kelimeler.csv` | Veriden çıkan ortak kelimeler |
| `sablon_sonuclari.csv`, `figures/sablon_katki.png`, `figures/sablon_kural_confusion.png` | Şablonun tek başına ve destek olarak başarısı |
| `sozluk_duzeltme_ornekleri.csv` | En sık yapılan düzeltmeler |
| `sozluk_sonuclari.csv`, `figures/sozluk_oncesi_sonrasi.png` | Düzeltmenin metne ve sınıflandırmaya etkisi |
| `data/ocr_duzeltilmis.csv` | Düzeltilmiş metinler |

Sonraki: **08_gurultu_giderme.ipynb** (istek 6)